Documet a correct implementation of the label generation Nov. 14

In [1]:
import random
import numpy as np

In [2]:
def random_mixed_state_uniform_radius(D):
    psi = np.random.uniform(-1, 1, D) + 1.j * np.random.uniform(-1, 1, D)
    psi = psi/np.linalg.norm(psi)
    rho = np.tensordot(psi, np.conjugate(psi), axes = 0)
    tensproduct = np.tensordot(psi, np.conjugate(psi), axes = 0)
    mu = np.random.uniform(0,1)
    return (1-mu) * np.identity(tensproduct[0].size) / D + mu * tensproduct

In [3]:
si = np.array([[1,0], [0,1]])
sx = np.array([[0,1] ,[1,0]])
sy = np.array([[0,-1j], [1j,0]])
sz = np.array([[1,0], [0,-1]])

pauli = [si, sx, sy, sz]

def numberToBase(n, b):
    if n == 0:
        return [0]
    digits = []
    while n:
        digits.append(int(n % b))
        n //= b
    return digits[::-1]

# stabilizer norm (density matrix -> real number)

def get_sn(rho, qubits):
    """
    Compute stabilizer norm using proper Pauli trace method.
    Fixed according to CLAUDE.md instructions.
    """
    if qubits == 1:
        
        X = np.array([[0,1],[1,0]], dtype=np.complex128)
        Y = np.array([[0,-1j],[1j,0]], dtype=np.complex128)
        Z = np.array([[1,0],[0,-1]], dtype=np.complex128)
        
        rx = float(np.trace(rho @ X).real)
        ry = float(np.trace(rho @ Y).real)
        rz = float(np.trace(rho @ Z).real)
        
        sn = abs(rx) + abs(ry) + abs(rz)
        return sn
    ## Sep.15 Update: sum of absolute values of all non-identity Pauli trace
    # manual extraction basis  
    # 枚举 其实还挺快的
    elif qubits == 2:
        I = np.array([[1,0],[0,1]], dtype=np.complex128)
        X = np.array([[0,1],[1,0]], dtype=np.complex128)
        Y = np.array([[0,-1j],[1j,0]], dtype=np.complex128)
        Z = np.array([[1,0],[0,-1]], dtype=np.complex128)
        paulis = [I, X, Y, Z]
        sn = 0.0
        # traverse all possible combinatioms 
        for i in range(4):
            for j in range(4):
                if i == 0 and j == 0:
                    continue  # skip I dot I cuz no contribution
                sig = np.kron(paulis[i], paulis[j])
                # Kronecker product
                sn += abs(np.trace(rho @ sig))
        return sn/4
    else:
        p = int(np.log2(np.size(rho[0])))
        dim = 2**p
        dim2 = 4**p
        
        a0 = 0
        for no in range(dim2):
            ntb = numberToBase(no, 4)
            op_no = np.pad(ntb, (p-len(ntb), 0), 'constant')
            op = [[1]]
            for i in range(p):
                op = np.kron(op,pauli[op_no[i]])
            a0 = np.array(a0+op)
        a0 = a0/dim
        
        aulist = []
        for no in range(dim2):
            ntb = numberToBase(no, 4)
            op_no = np.pad(ntb, (p-len(ntb), 0), 'constant')
            op = [[1]]
            for i in range(p):
                op = np.kron(op,pauli[op_no[i]])
            aulist.append( np.dot(np.dot(op, a0), np.matrix(op).getH()) )
        
        wigner = [np.trace(np.dot(aulist[n], rho))/dim for n in range(dim2)]
        return np.real(np.sum(np.absolute(wigner)-wigner)/2)

In [7]:
rho = random_mixed_state_uniform_radius(8)
sn = get_sn(rho, 3)
print(rho)
print(sn)

[[ 0.1330033 -4.18351952e-19j  0.02334724+7.72430176e-03j
   0.00959767-3.70612735e-02j  0.02650101-1.20561555e-02j
  -0.03365807+2.35514971e-02j -0.03028596-6.32314101e-03j
   0.01199839-1.27104678e-02j  0.0261777 +3.25651848e-02j]
 [ 0.02334724-7.72430176e-03j  0.11110315+1.50070851e-19j
  -0.0016423 -2.48065196e-02j  0.0138792 -1.28382486e-02j
  -0.01594693+2.13851451e-02j -0.01996152+2.27913482e-03j
   0.00480464-1.02835307e-02j  0.02278133+1.47374753e-02j]
 [ 0.00959767+3.70612735e-02j -0.0016423 +2.48065196e-02j
   0.13383626-5.66820897e-19j  0.01851521+2.28798136e-02j
  -0.03157906-2.69707274e-02j -0.0014875 -3.12420161e-02j
   0.01548002+8.52093501e-03j -0.02523558+3.38722256e-02j]
 [ 0.02650101+1.20561555e-02j  0.0138792 +1.28382486e-02j
   0.01851521-2.28798136e-02j  0.11751715+1.93223393e-19j
  -0.03105162+5.76587261e-03j -0.01918096-1.40667290e-02j
   0.01244292-5.07493133e-03j  0.00795161+3.11228964e-02j]
 [-0.03365807-2.35514971e-02j -0.01594693-2.13851451e-02j
  -0.03157